In [1]:
pip install ultralytics torch torchvision


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [38]:
import torch

BEST_PT = r"D:\Users\eya20\Downloads\best (6).pt"
DATASET_DIR = r"D:\Users\eya20\Downloads\Finall.v2i.yolov8"
DATA_YAML = r"D:\Users\eya20\front lwm\bo-lwm\data.yaml"
TEST_IMAGE = r"D:\Users\eya20\scrapping\waste_dataset\old_kitchen_appliances\old_kitchen_appliances_bing_000434.jpg"

ckpt = torch.load(BEST_PT, map_location="cpu", weights_only=False)
print("Epoch:", ckpt.get("epoch"))
print("Best fitness:", round(float(ckpt.get("best_fitness", 0)), 5))

Epoch: 43
Best fitness: 0.56335


In [39]:
from ultralytics import YOLO

model = YOLO(BEST_PT)
model.predict(
    TEST_IMAGE,
    conf=0.50,
    iou=0.45,
    imgsz=640,
    save=True,
)


image 1/1 D:\Users\eya20\scrapping\waste_dataset\old_kitchen_appliances\old_kitchen_appliances_bing_000434.jpg: 512x640 1 American double fridge, 1 Microwave, 309.2ms
Speed: 0.0ms preprocess, 309.2ms inference, 0.0ms postprocess per image at shape (1, 3, 512, 640)
Results saved to runs\detect\predict10


[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'American double fridge', 1: 'BBQ', 2: 'Bike', 3: 'Bin', 4: 'Book', 5: 'Car Battery', 6: 'Chest freezer', 7: 'Coffee machine', 8: 'Cooker hood', 9: 'Dinning chair', 10: 'Dishwasher', 11: 'Dvd Players - Sky Box', 12: 'Exercice bike', 13: 'External door', 14: 'Fan', 15: 'Footstool', 16: 'Garden chair', 17: 'Garden table', 18: 'Hob Cooker', 19: 'Iron', 20: 'Ironing board', 21: 'Kettle', 22: 'Keyboard', 23: 'Lamp', 24: 'Laptop', 25: 'Lawnmower', 26: 'Microwave', 27: 'Office printer', 28: 'Oven', 29: 'Paint cans', 30: 'Parasol', 31: 'Plant', 32: 'Pool', 33: 'Pot', 34: 'Radiator', 35: 'Radio', 36: 'Sink', 37: 'Standard fridge freezer', 38: 'Suitcase', 39: 'TV', 40: 'Toilet', 41: 'Waste bag', 42: 'Window'}
 obb: None
 orig_img: array([[[220, 220, 227],
         [217, 220, 226],
         [213, 218, 224],
         ...,
         [246, 249, 254],


In [40]:
results = model.predict(
    TEST_IMAGE,
    conf=0.50,
    iou=0.45,
    imgsz=640,
    save=True,
)

r = results[0]
print("Number of detections:", len(r.boxes))

for box in r.boxes:
    cls_id = int(box.cls[0])
    conf = float(box.conf[0])
    print(f"  {r.names[cls_id]}: {conf:.2%}")

image 1/1 D:\Users\eya20\scrapping\waste_dataset\old_kitchen_appliances\old_kitchen_appliances_bing_000434.jpg: 512x640 1 American double fridge, 1 Microwave, 366.4ms
Speed: 4.0ms preprocess, 366.4ms inference, 1.7ms postprocess per image at shape (1, 3, 512, 640)
Results saved to runs\detect\predict10
Number of detections: 2
  American double fridge: 90.42%
  Microwave: 86.96%


In [41]:
results[0].show()   # opens image with boxes drawn

In [20]:
import os, yaml

with open(DATA_YAML) as f:
    data = yaml.safe_load(f)

base = data["path"]
print("Classes (nc):", data["nc"])
print("Dataset base:", base)
print("val images:", os.path.isdir(os.path.join(base, "valid", "images")))
print("test images:", os.path.isdir(os.path.join(base, "test", "images")))

Classes (nc): 43
Dataset base: D:/Users/eya20/Downloads/Finall.v2i.yolov8
val images: True
test images: True


In [21]:
from ultralytics import YOLO
import yaml

model = YOLO(BEST_PT)

val = model.val(data=DATA_YAML, split="val", imgsz=640)
test = model.val(data=DATA_YAML, split="test", imgsz=640)

print("\n===== FINAL METRICS =====")
print(f"VAL  -> P={val.box.mp:.4f}  R={val.box.mr:.4f}  mAP50={val.box.map50:.4f}  mAP50-95={val.box.map:.4f}")
print(f"TEST -> P={test.box.mp:.4f}  R={test.box.mr:.4f}  mAP50={test.box.map50:.4f}  mAP50-95={test.box.map:.4f}")

with open(DATA_YAML) as f:
    class_names = yaml.safe_load(f).get("names", {})

if isinstance(class_names, dict):
    names = [class_names[k] for k in sorted(class_names.keys(), key=lambda x: int(x))]
else:
    names = class_names

print("\n===== PER-CLASS (val) =====")
for i, name in enumerate(names):
    if i < len(val.box.p):
        print(f"{name:30} P={val.box.p[i]:.3f}  R={val.box.r[i]:.3f}  mAP50={val.box.ap50[i]:.3f}")

Ultralytics 8.3.31  Python-3.12.6 torch-2.5.1+cpu CPU (Intel Core(TM) i5-1035G1 1.00GHz)
YOLO11s summary (fused): 239 layers, 9,429,441 parameters, 0 gradients, 21.4 GFLOPs


val: Scanning D:\Users\eya20\Downloads\Finall.v2i.yolov8\valid\labels... 1936 images, 407 backgrounds, 1 corrupt: 100%|██████████| 1937/1937 [00:10<00:00, 177.99it/s]

val: WARNING  D:\Users\eya20\Downloads\Finall.v2i.yolov8\valid\images\dinglilighting-rustic-floor-lamps-for-living-room-modern-tall-pole-light-with-adjustable-reading-light-vintage-standing-lamp-for-contemporary-mid-century-bedroo-8223_jpg.rf.9818b99a43d39c948d150bd8c65f33ae.jpg: ignoring corrupt image/label: [Errno 2] No such file or directory: 'D:\\Users\\eya20\\Downloads\\Finall.v2i.yolov8\\valid\\images\\dinglilighting-rustic-floor-lamps-for-living-room-modern-tall-pole-light-with-adjustable-reading-light-vintage-standing-lamp-for-contemporary-mid-century-bedroo-8223_jpg.rf.9818b99a43d39c948d150bd8c65f33ae.jpg'


val: New cache created: D:\Users\eya20\Downloads\Finall.v2i.yolov8\valid\labels.cache
WARNING  Box and segment counts should be equal, but got len(segments) = 10, len(boxes) = 2323. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95):   2%|▏         | 3/121 [00:22<14:44,  7.49s/it]

...

val: Scanning D:\Users\eya20\Downloads\Finall.v2i.yolov8\test\labels... 1936 images, 229 backgrounds, 1 corrupt: 100%|██████████| 1937/1937 [00:06<00:00, 278.07it/s]

val: WARNING  D:\Users\eya20\Downloads\Finall.v2i.yolov8\test\images\depuley-modern-globe-led-floor-lamps-for-living-room-dllt-standing-lamps-with-5-lights-for-bedroom-tall-pole-tree-accent-lighting-for-mid-century-contemporary-h-3116_jpg.rf.91ef6d9d8f09a59290fc9ce9f0a39a6f.jpg: ignoring corrupt image/label: [Errno 2] No such file or directory: 'D:\\Users\\eya20\\Downloads\\Finall.v2i.yolov8\\test\\images\\depuley-modern-globe-led-floor-lamps-for-living-room-dllt-standing-lamps-with-5-lights-for-bedroom-tall-pole-tree-accent-lighting-for-mid-century-contemporary-h-3116_jpg.rf.91ef6d9d8f09a59290fc9ce9f0a39a6f.jpg'


val: New cache created: D:\Users\eya20\Downloads\Finall.v2i.yolov8\test\labels.cache


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 121/121 [11:12<00:00,  5.56s/it]


                   all       1936       2539      0.815      0.727      0.803      0.575
American double fridge         19         19      0.938        0.8      0.958      0.823
                   BBQ         13         13      0.874      0.846      0.858      0.675
                  Bike         39         48      0.943      0.708      0.816      0.629
                   Bin        123        226      0.832      0.569      0.741      0.449
                  Book          5         70       0.69      0.186      0.497      0.234
           Car Battery         11         13      0.773      0.923      0.946      0.714
         Chest freezer         10         10      0.806        0.8      0.865      0.802
        Coffee machine         99        112      0.869      0.832      0.923      0.642
           Cooker hood         10         10      0.619        0.7      0.723      0.618
         Dinning chair        114        238      0.865      0.674       0.76      0.539
            Dishwashe